# ClimaCity Paris -- Session 5
## Machine Learning avec MLlib, optimisation et bilan d'architecture

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Prérequis** : Avoir complété les Jours 1 et 2 -- la table Delta
`data/output/delta/disponibilite` doit être présente.

---

Ce notebook couvre l'intégralité du Jour 3 du projet ClimaCity Paris.

- **Partie 1 -- Matin (3h30)** : Machine Learning distribué avec MLlib.
  Construction de features, clustering des stations (K-Means), modèle de
  prédiction du taux d'occupation (GBTRegressor), validation croisée,
  et suivi des expériences avec MLflow.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne.  
> Les cellules `# [CORRECTION]` proposent une solution.


---
## Section 0 -- Configuration


In [ ]:
from pathlib import Path
import time, warnings
warnings.filterwarnings("ignore")

# ── Chemins ──────────────────────────────────────────────────────────────────
DATA_DIR         = Path("../data")
OUTPUT_DIR       = DATA_DIR / "output"
DELTA_DISPONIBLE = OUTPUT_DIR / "delta" / "disponibilite"
STATIONS_CSV     = DATA_DIR / "velib" / "stations_info.csv"
MODELS_DIR       = OUTPUT_DIR / "models"
MLFLOW_DIR       = OUTPUT_DIR / "mlruns"

for p in [DELTA_DISPONIBLE]:
    assert p.exists(), f"Fichier manquant : {p} -- relancez les Jours 1 et 2"
for p in [MODELS_DIR, MLFLOW_DIR]:
    p.mkdir(parents=True, exist_ok=True)

APP_NAME      = "ClimaCity-Paris-Jour3"
SHUFFLE_PARTS = 8
SEED          = 42


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.driver.memory", "6g")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- http://localhost:4040")


In [ ]:
# Chargement de la table Delta consolidée
df = (
    # TODO : Charger la table Spark du dossier DELTA_DISPONIBLE
)
df.count()   # force le cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()


---
# PARTIE 1 -- Apprentissage automatique avec MLlib

## 1.1 Les trois abstractions fondamentales de MLlib

MLlib repose sur trois interfaces qui permettent de construire des pipelines
ML reproductibles et composables.

### Transformer

Un `Transformer` prend un DataFrame en entrée et retourne un DataFrame enrichi.
Il implémente la méthode `transform(df)`.  
Exemples : `VectorAssembler`, `StandardScaler` (après fit), `Binarizer`.

### Estimator

Un `Estimator` est un algorithme qui s'entraîne sur des données.
Il implémente la méthode `fit(df)` et retourne un `Transformer` (le modèle ajusté).  
Exemples : `GBTRegressor`, `KMeans`, `StandardScaler` (avant fit).

### Pipeline

Un `Pipeline` enchaîne une liste ordonnée de `Transformer` et d'`Estimator`.
Lorsqu'on appelle `pipeline.fit(df)`, chaque étape est exécutée en séquence.
Le résultat est un `PipelineModel`, qui lui-même est un `Transformer`.

```
DataFrame d'entrée
      │
      ▼
  Étape 1 : VectorAssembler  (Transformer)  → features brutes
      │
      ▼
  Étape 2 : StandardScaler   (Estimator)    → calcule mean/std sur train
      │         ↓ fit → StandardScalerModel (Transformer)
      ▼
  Étape 3 : GBTRegressor     (Estimator)    → entraîne le modèle
                ↓ fit → GBTRegressionModel  (Transformer)
      │
      ▼
DataFrame de sortie (avec colonne "prediction")
```

La clé du Pipeline : on appelle `fit()` **une seule fois** sur les données
d'entraînement. Le `PipelineModel` résultant peut être appliqué sur les données
de test **sans risque de fuite d'information** entre train et test.


---
## 1.2 Construction des features

Un bon modèle commence par de bonnes features. Nous allons construire
un vecteur de features à partir des informations disponibles :
contexte temporel, conditions météorologiques, et historique récent de la station.


In [ ]:
from pyspark.sql.functions import (
    col, when, sin, cos, lit, lag, avg as spark_avg,
    to_timestamp, unix_timestamp, round as spark_round,
    dayofyear
)
from pyspark.sql.window import Window

# ── 1. Features temporelles cycliques ────────────────────────────────────────
# L'heure 23 et l'heure 0 sont proches dans le temps mais éloignées en valeur
# brute. On encode l'heure et le jour de semaine sur un cercle (sin/cos)
# pour que le modèle perçoive leur nature cyclique
#
# Chaque information ('heure', 'jour_sem', 'mois') est encodé par un sinus + un cosinus
# Le cercle est divisé en parts égales proportionnelles au nombre de valeurs

PI = 3.14159265358979

df_feat = (
    # Ajouter à `df` les colonnes correspondant aux données temporelles
)

# ── 2. Features météo normalisées ────────────────────────────────────────────
# Les valeurs manquantes sont imputées par la médiane (calculée sur le train).
# Ici on les remplace provisoirement par des constantes raisonnables (à vous de les choisir)
#
# cf. la fonction `coalesce` de la bibliothèque `functions` de Spark
# pour traiter les valeurs manquantes.

df_feat = (
    # TODO : ajouter à `df_feat` les données météorologiques normalisées
)

# ── 3. Features de lag (historique récent de la station) ─────────────────────
# Le taux d'occupation à t-1 et t-4 (environ 15 et 60 minutes avant)
# sont souvent les features les plus prédictives pour ce type de série.

fenetre_station = (
    Window
    .partitionBy("station_id")
    .orderBy("horodatage")
)

# cf. les fonctions `lag` et `over` pour calculer l'historique
df_feat = (
    # TODO : Ajouter à `df_feat` les colonnes pour 
    # - le taux d'occupation à t-1
    # - le taux d'occupation à t-4
    # - la moyenne du taux d'occupation entre t-4 et t-1
)

# ── 4. Variable cible : taux_occupation à t+4 (≈ 1 heure dans le futur) ──────

# La variable cible est celle que l'on cherche à prédire
# cf. la fonction `lead`
df_feat = df_feat.withColumn(
    "cible",
    # TODO : Ajouter la valeur de la nouvelle colonne qui indique le taux d'occupation à +1 heure 
)

# ── 5. Suppression des lignes incomplètes (lag/lead génère des nulls) ──────────

# TODO : Supprimer de `df_feat` les lignes comportant des `None` -> nouveau DataFrame `df_ml`

print(f"Lignes disponibles pour le ML : {df_ml.count():,}")
print(f"Features ({len(FEATURES)}) : {FEATURES}")


## 1.3 Séparation du jeu de données

Créer un jeu de données pour l'entraînement et un jeu de données pour la validation/test

In [ ]:
# ── 6. Split train / test ──────────────────────────────────────────────────────
# IMPORTANT : on split dans le temps, pas aléatoirement.
# Un split aléatoire sur des séries temporelles entraîne une fuite d'information :
# le modèle voit des données du futur pendant l'entraînement.
# Par exemple, si les données sont disponibles : 2022 = train, 2023 = test
# Sinon, séparer selon un ratio 85/15
df_train = # ??
df_test  = # ??

print(f"Train (2022) : {df_train.count():,} lignes")
print(f"Test  (2023) : {df_test.count():,} lignes")
print(f"Ratio        : {df_train.count() / df_ml.count():.1%} / {df_test.count() / df_ml.count():.1%}")

# Mise en cache des deux splits -- ils seront accédés plusieurs fois
df_train.cache(); df_train.count()
df_test.cache();  df_test.count()


## 1.4 Clustering des stations avec K-Means

Avant de prédire la disponibilité individuelle de chaque station, nous allons
**regrouper les stations par profil comportemental** : certaines sont saturées
le matin (quartiers résidentiels qui "envoient" des cyclistes vers le centre),
d'autres le soir, d'autres ont un usage homogène sur la journée.

Ce clustering a deux utilités :
1. Comprendre la géographie fonctionnelle du réseau.
2. Ajouter l'identifiant de cluster comme feature dans le modèle de régression.

### Construction du profil de station

Le vecteur de profil d'une station est son **taux d'occupation moyen à chaque
heure de la journée** -- un vecteur de dimension 24.


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import ClusteringEvaluator

# ── Profil horaire moyen par station ─────────────────────────────────────────
# Pivot : une ligne par station, une colonne par heure

df_profil = (
    df_ml
    .groupBy("station_id")
    .pivot("heure", list(range(24)))
    .agg(spark_round(spark_avg("taux_occupation"), 4))
)

# Renommage des colonnes pivot (0 -> h00, 1 -> h01, ...)
for h in range(24):
    df_profil = df_profil.withColumnRenamed(str(h), f"h{h:02d}")

colonnes_profil = [f"h{h:02d}" for h in range(24)]
df_profil = df_profil.dropna(subset=colonnes_profil)

print(f"Stations avec profil complet : {df_profil.count()}")
df_profil.show(5, truncate=True)


In [ ]:
# ── Méthode du coude : inertie (WSSSE) en fonction de k ──────────────────────
# On entraîne un K-Means pour k = 2..8 et on trace l'inertie.
# Le "coude" dans la courbe indique le bon compromis.

# VectorAssembler transforme un « tableau » (un ensemble de colonnes) en un vecteur
# Transformation essentielle pour fournir des données à l'algorithme de clusterisation
assembler_profil = VectorAssembler(
    inputCols=colonnes_profil,
    outputCol="features_profil"
)

# StandardScaler normalise les valeurs entre [-1, +1]
# en formant une distribution appelée « réduite centrée »
# centrée autour de la moyenne, réduite à 1 écart-type
scaler_profil = StandardScaler(
    inputCol="features_profil",
    outputCol="features_scaled",
    withStd=True, withMean=True
)

inertie_par_k = {}

# Exécuter l'algorithme K-Means en essayant plusieurs valeurs pour le nombre de clusters
#
for k in range(2, 9):
    kmeans = KMeans(
        # TODO : Comment configurer l'algorithme K-Means
    )
    # Avec Pipeline, créer un processus assemblant trois étapes :
    # `assemble_profil`, `scaler_profil` et `kmeans`
    # Puis entraîner le modèle
    # TODO : créer le modèle `model_km`
    
    # WSSSE = Within Set Sum of Squared Errors (inertie intra-cluster)
    # Le Within Set Sum of Squared Errors (WSSSE) est la somme des carrés des distances 
    # entre chaque point de données et le centroïde de son cluster dans un algorithme de K-Means.
    # Le centroïde est aussi appelé barycentre ou centre de gravité
    wssse = model_km.stages[-1].summary.trainingCost
    inertie_par_k[k] = round(wssse, 2)
    print(f"  k={k}  inertie={wssse:,.1f}")

print("\nTableau récapitulatif :")
for k, w in inertie_par_k.items():
    barre = "=" * int(w / max(inertie_par_k.values()) * 40)
    print(f"  k={k}  {w:>12,.1f}  {barre}")


In [ ]:
# ── Entraînement final avec le k retenu ───────────────────────────────────────
# Choisissez le k correspondant au "coude" observé ci-dessus.
# En pratique sur ce jeu de données, k=4 ou k=5 est souvent pertinent.
K_RETENU = # Valeur optimale

# TODO : Récréer le modèle final avec le  nombre de clusters optimal

# Construction d'un DataFrame issu de la clusterisation
# et ne contenant que l'identifiant de la station et son clusterd'appartenance
df_clusters = model_km_final.transform(df_profil).select("station_id", "cluster")

# Taille de chaque cluster
print(f"Répartition des {df_clusters.count()} stations en {K_RETENU} clusters :")
# TODO : Afficher la taille de chaque cluster

In [ ]:
# ── Interprétation : profil horaire moyen par cluster ────────────────────────

# 1. Créer un DataFrame par jointure entre `df_profil` et `df_clusters` sur l'identifiant de la station
# 2. Calculer le taux d'occupation moyen par heure pour chaque cluster, trié par cluster
print("Taux d'occupation moyen par cluster et par heure (6h, 9h, 12h, 18h, 22h) :")
(
    # TODO 2.
)


## Folium

Folium s'appuie sur les points forts de l'écosystème Python en termes d'analyse de données et sur ceux de la bibliothèque JS Leaflet our l'affichage cartographique. Manipulez vos données en Python, puis visualisez-les dans une carte via Folium.

### Concepts

Folium facilite la visualisation des données manipulées en Python sur une carte interactive. Il permet à la fois la liaison de données avec une carte géographique pour les visualisations choroplèthes, et de passer comme marqueurs de graphismes riche  vectoriels/binaires/HTML.

La bibliothèque dispose d'un certain nombre de jeux de tuiles intégrés d'OpenStreetMap, Mapbox, etc. Elle prend aussi en charge les jeux de tuiles personnalisés. 

Folium prend en charge les superpositions Image, Video, GeoJSON et TopoJSON et dispose d'un nombre de couches vectorielles intégrées.


In [1]:
# ── Visualisation sur carte Folium ─────────────────────────────────────────
import folium
import pandas as pd

PALETTE = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12",
           "#9b59b6", "#1abc9c", "#e67e22"]


# Créer un DataFrame `df_carte` par jointure en `df_clusters` et le fchier des stations
# Ne garder de ce dernier que : "station_id", "lat", "lon", "name"
# 
# Le fichier des stations est lu avec pandas
# et le DataFrame résultant est de nouveau converti en Pandas

# Création d'une carte avec Folium
carte = folium.Map(location=[48.8566, 2.3522], zoom_start=12, tiles="CartoDB positron")

for _, row in df_carte.iterrows():
    # Pour toutes les stations,
    # afficher un marqueur sous forme de cercle, aux coordonnées de géolocalisation données
    # avec les paramètres suivants :
    # - une couleur par cluster
    # - un rayon de 5 pour le cercle
    # - le cercle est plein, avec une opacité de 0.8
    # - chaque point doit être associé ç une fenêtre popup indiquant le numéro du cluster
    # - afficher également un tooltip
    #
    # cf. Folium :
    # - CircleMarker
    # - la méthode `add_to`
    couleur = PALETTE[int(row["cluster"]) % len(PALETTE)]

# Légende textuelle ajoutée à la carte
legende_html = "<div style='position:fixed;bottom:30px;left:30px;z-index:1000;background:white;padding:10px;border:1px solid #ccc;font-size:12px'>"
for i in range(K_RETENU):
    legende_html += (
        f"<div><span style='display:inline-block;width:14px;height:14px;"
        f"background:{PALETTE[i]};border-radius:50%;margin-right:6px'></span>"
        f"Cluster {i}</div>"
    )
legende_html += "</div>"
carte.get_root().html.add_child(folium.Element(legende_html))

chemin_carte = OUTPUT_DIR / "carte_clusters_velib.html"
carte.save(str(chemin_carte))
print(f"Carte sauvegardée : {chemin_carte}")
print("Ouvrez ce fichier dans votre navigateur pour visualiser les clusters.")

# Affichage inline dans Jupyter
carte


ModuleNotFoundError: No module named 'folium'

In [ ]:
# Ajout du cluster comme feature pour le modèle de régression
df_ml = df_ml.join(df_clusters, on="station_id", how="left").fillna(0, subset=["cluster"])
df_train = df_train.join(df_clusters, on="station_id", how="left").fillna(0, subset=["cluster"])
df_test  = df_test.join(df_clusters, on="station_id", how="left").fillna(0, subset=["cluster"])

FEATURES = FEATURES + ["cluster"]
print(f"Features après ajout du cluster ({len(FEATURES)}) : {FEATURES}")


---
## 1.5 Modèle de régression : GBTRegressor

Le **Gradient Boosted Tree Regressor** est un algorithme d'ensemble qui construit
séquentiellement des arbres de décision, chacun corrigeant les erreurs du précédent.
C'est l'un des meilleurs algorithmes pour les données tabulaires avec des features
hétérogènes -- exactement notre cas.

MLlib l'implémente de façon distribuée : chaque arbre est construit en parallèle
sur les partitions du DataFrame.


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# ── Construction du Pipeline ──────────────────────────────────────────────────

# 1. Créer une représentation sous forme de vecteur des features
# Dans une colonne `features_brutes`
# Supprimer les lignes invalides

# 2. Normaliser les valeurs de la colonne `features_brutes`
# en créant une nouvelle colonne `features`
# Les valeurs sont centrées réduites

# 3. Créer un modèle de régression
# - qui prend entrée le vecteur `features`
# - qui a pour sortie la colonne `cible`
# - la colonne de prédiction s'appelle `prediction`
# Nous voulons les paramètres suivants :
# - 50 arbres de décision
# - un pas d'apprentissaqge de 0.1
# - ne pas oublier d'initaliser le processus aléatoire
gbt = GBTRegressor(
    # TODO : Créer le modèle de régression
)

# 3. Créer le pipeline complet

# ── Entraînement ──────────────────────────────────────────────────────────────
print("Entraînement du GBTRegressor (peut prendre 1-2 minutes)...")
t0 = time.perf_counter()
model_gbt = pipeline_gbt.fit(df_train)
t_fit = time.perf_counter() - t0
print(f"Entraînement terminé en {t_fit:.1f} s")


In [ ]:
# ── Évaluation sur le test set ────────────────────────────────────────────────
evaluator_rmse = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="r2"
)
evaluator_mae = RegressionEvaluator(
    labelCol="cible", predictionCol="prediction", metricName="mae"
)

df_pred = model_gbt.transform(df_test)

rmse = evaluator_rmse.evaluate(df_pred)
r2   = evaluator_r2.evaluate(df_pred)
mae  = evaluator_mae.evaluate(df_pred)

print(f"Métriques sur le test set (2023) :")
print(f"  RMSE (Root Mean Squared Error) : {rmse:.4f}")
print(f"  MAE  (Mean Absolute Error)     : {mae:.4f}")
print(f"  R²   (coefficient de détermination) : {r2:.4f}")
print()
print("Interprétation :")
print(f"  En moyenne, le modèle se trompe de ±{mae:.3f} sur le taux d'occupation (0-1).")
print(f"  Soit ±{mae*100:.1f} points de pourcentage.")


In [ ]:
# ── Importance des features ────────────────────────────────────────────────────
# Le GBT calcule l'importance de chaque feature (réduction d'impureté moyenne)
model_gbt_stage = model_gbt.stages[-1]   # le GBTRegressionModel
importances     = model_gbt_stage.featureImportances.toArray()

df_importance = (
    pd.DataFrame({
        "feature"   : FEATURES,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Importance des features (top 10) :")
print(f"{'Rang':<5} {'Feature':<25} {'Importance':>12}  Barre")
print("-" * 65)
for i, row in df_importance.head(10).iterrows():
    barre = "█" * int(row["importance"] * 200)
    print(f"  {i+1:<3} {row['feature']:<25} {row['importance']:>12.4f}  {barre}")


In [ ]:
# ── Analyse des erreurs : où le modèle se trompe-t-il le plus ? ───────────────
df_erreurs = (
    df_pred
    .withColumn("erreur_abs", F.abs(col("prediction") - col("cible")))
    .withColumn("erreur_rel",
        F.abs(col("prediction") - col("cible")) / (col("cible") + F.lit(0.01))
    )
)

print("Erreur absolue moyenne par heure de la journée :")
(
    df_erreurs
    .groupBy("heure")
    .agg(
        spark_round(spark_avg("erreur_abs"), 4).alias("mae_horaire"),
        F.count("*").alias("n")
    )
    .orderBy("heure")
    .show(24)
)


In [ ]:
# [EXERCICE]
# Calculez l'erreur absolue moyenne (MAE) par cluster de station.
# Quel cluster est le plus difficile à prédire ? Pourquoi, à votre avis ?
#
# Indice : groupBy("cluster").agg(...)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :


In [ ]:
# [CORRECTION]
print("MAE par cluster de station :")
(
    df_erreurs
    .groupBy("cluster")
    .agg(
        spark_round(spark_avg("erreur_abs"), 4).alias("mae_cluster"),
        spark_round(spark_avg("erreur_rel"), 4).alias("mare_cluster"),
        F.count("*").alias("n_predictions")
    )
    .orderBy(F.desc("mae_cluster"))
    .show()
)
# Les stations de cluster à fort usage (très vides le matin, très pleines le soir)
# sont généralement plus difficiles à prédire car leur dynamique est plus abrupte.


---
## 1.6 Validation croisée et optimisation des hyperparamètres

Nos hyperparamètres actuels (`maxIter=50`, `maxDepth=5`, `stepSize=0.1`)
ont été choisis arbitrairement. Le `CrossValidator` de MLlib permet
d'évaluer systématiquement plusieurs combinaisons et de choisir la meilleure.

> **Attention** : la validation croisée sur des séries temporelles est
> délicate. Le `CrossValidator` de MLlib effectue un K-Fold standard
> (mélanges aléatoires des données), ce qui peut créer de la fuite
> d'information temporelle. Pour un usage production, on préférerait
> une validation en fenêtre glissante (*time series split*).
> Dans le contexte de ce cours, le K-Fold est acceptable pour comprendre
> le mécanisme.


In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# ── Grille d'hyperparamètres ──────────────────────────────────────────────────
param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth,  [3, 5])
    .addGrid(gbt.maxIter,   [30, 50])
    .addGrid(gbt.stepSize,  [0.05, 0.1])
    .build()
)
print(f"Combinaisons à évaluer : {len(param_grid)}")

# ── CrossValidator ────────────────────────────────────────────────────────────
cv = CrossValidator(
    estimator=pipeline_gbt,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_rmse,
    numFolds=3,
    seed=SEED,
    parallelism=2    # évalue 2 combinaisons en parallèle
)

# On entraîne sur un sous-échantillon pour limiter le temps de calcul en cours
df_train_sample = df_train.sample(fraction=0.3, seed=SEED).cache()
df_train_sample.count()
print(f"Sous-échantillon train : {df_train_sample.count():,} lignes")

print("\nValidation croisée en cours (peut prendre 3-5 minutes)...")
t0 = time.perf_counter()
cv_model = cv.fit(df_train_sample)
t_cv = time.perf_counter() - t0
print(f"Terminé en {t_cv:.1f} s")


In [ ]:
# ── Résultats de la grille ────────────────────────────────────────────────────
avg_metrics = cv_model.avgMetrics
params_list = [
    {p.name: v for p, v in zip(param_grid[0].keys(), combo.values())}
    for combo in param_grid
]

resultats_cv = pd.DataFrame(params_list)
resultats_cv["rmse_cv"] = avg_metrics
resultats_cv = resultats_cv.sort_values("rmse_cv").reset_index(drop=True)

print("Résultats du CrossValidator (triés par RMSE croissant) :")
print(resultats_cv.to_string(index=False))

# Meilleurs hyperparamètres
meilleur = resultats_cv.iloc[0]
print(f"\nMeilleure combinaison :")
for col_name in ["maxDepth", "maxIter", "stepSize", "rmse_cv"]:
    print(f"  {col_name:<12} : {meilleur[col_name]}")


In [ ]:
# Évaluation du meilleur modèle sur le test set complet
df_pred_best = cv_model.bestModel.transform(df_test)

rmse_best = evaluator_rmse.evaluate(df_pred_best)
r2_best   = evaluator_r2.evaluate(df_pred_best)
mae_best  = evaluator_mae.evaluate(df_pred_best)

print("Comparaison modèle initial vs meilleur modèle CV :")
print(f"{'Métrique':<8}  {'Initial':>12}  {'CV Best':>12}  {'Gain':>10}")
print("-" * 46)
print(f"{'RMSE':<8}  {rmse:>12.4f}  {rmse_best:>12.4f}  {rmse - rmse_best:>+10.4f}")
print(f"{'MAE':<8}  {mae:>12.4f}  {mae_best:>12.4f}  {mae - mae_best:>+10.4f}")
print(f"{'R²':<8}  {r2:>12.4f}  {r2_best:>12.4f}  {r2_best - r2:>+10.4f}")


# 1.7 Suivi des expériences avec MLflow

MLflow est un outil de tracking d'expériences ML : il enregistre
hyperparamètres, métriques, artefacts (modèles, graphiques) et permet
de comparer les runs dans une interface web.

```bash
# Pour ouvrir l'interface MLflow dans un terminal séparé :
mlflow ui --backend-store-uri data/output/mlruns --port 5000
# Puis ouvrir http://localhost:5000 dans le navigateur
```


In [ ]:
import mlflow
import mlflow.spark

# Configuration du répertoire de tracking
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR.resolve()}")
mlflow.set_experiment("ClimaCity-Paris-GBT")

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Expérience          : ClimaCity-Paris-GBT")


In [ ]:
def evaluer_et_logger(
    df_train_fit, df_test_eval,
    max_depth: int, max_iter: int, step_size: float,
    nom_run: str
) -> dict:
    """Entraîne un pipeline GBT, l'évalue et logue les résultats dans MLflow.

    Args:
        df_train_fit  : DataFrame d'entraînement.
        df_test_eval  : DataFrame de test.
        max_depth     : Profondeur maximale des arbres.
        max_iter      : Nombre d'arbres (iterations).
        step_size     : Taux d'apprentissage.
        nom_run       : Nom du run MLflow.

    Returns:
        Dictionnaire avec les métriques calculées sur le test set.
    """
    with mlflow.start_run(run_name=nom_run):
        # ── Paramètres ──
        mlflow.log_params({
            "max_depth" : max_depth,
            "max_iter"  : max_iter,
            "step_size" : step_size,
            "n_features": len(FEATURES),
            "n_train"   : df_train_fit.count(),
            "n_test"    : df_test_eval.count(),
        })

        # ── Entraînement ──
        gbt_run = GBTRegressor(
            featuresCol="features", labelCol="cible",
            predictionCol="prediction",
            maxIter=max_iter, maxDepth=max_depth,
            stepSize=step_size, seed=SEED
        )
        pipeline_run = Pipeline(stages=[assembler, scaler, gbt_run])
        t0    = time.perf_counter()
        model = pipeline_run.fit(df_train_fit)
        t_fit = time.perf_counter() - t0

        # ── Métriques ──
        preds = model.transform(df_test_eval)
        metriques = {
            "rmse"     : evaluator_rmse.evaluate(preds),
            "mae"      : evaluator_mae.evaluate(preds),
            "r2"       : evaluator_r2.evaluate(preds),
            "fit_time" : round(t_fit, 2),
        }
        mlflow.log_metrics(metriques)

        # ── Modèle ──
        mlflow.spark.log_model(model, artifact_path="pipeline_gbt")

        print(f"  [{nom_run}]  RMSE={metriques['rmse']:.4f}  "
              f"R²={metriques['r2']:.4f}  ({t_fit:.1f}s)")
        return metriques

# ── Plusieurs runs pour comparer ──────────────────────────────────────────────
print("Lancement des runs MLflow...")
configs = [
    {"max_depth": 3, "max_iter": 30, "step_size": 0.1,  "nom_run": "shallow-fast"},
    {"max_depth": 5, "max_iter": 50, "step_size": 0.1,  "nom_run": "medium-balanced"},
    {"max_depth": 5, "max_iter": 50, "step_size": 0.05, "nom_run": "medium-slow-lr"},
    {"max_depth": 7, "max_iter": 80, "step_size": 0.05, "nom_run": "deep-thorough"},
]

resultats_mlflow = []
for cfg in configs:
    m = evaluer_et_logger(df_train, df_test, **cfg)
    resultats_mlflow.append({"run": cfg["nom_run"], **m})

print("\nTableau comparatif :")
df_res = pd.DataFrame(resultats_mlflow).sort_values("rmse")
print(df_res.to_string(index=False))


In [ ]:
# ── Chargement du meilleur modèle depuis MLflow ───────────────────────────────
from mlflow.tracking import MlflowClient

client  = MlflowClient()
exp     = client.get_experiment_by_name("ClimaCity-Paris-GBT")
runs    = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.rmse ASC"],
    max_results=1
)
best_run = runs[0]

print(f"Meilleur run :")
print(f"  ID   : {best_run.info.run_id}")
print(f"  Nom  : {best_run.data.tags.get('mlflow.runName', 'N/A')}")
print(f"  RMSE : {best_run.data.metrics['rmse']:.4f}")
print(f"  R²   : {best_run.data.metrics['r2']:.4f}")

# Rechargement du modèle
model_uri     = f"runs:/{best_run.info.run_id}/pipeline_gbt"
model_recharge = mlflow.spark.load_model(model_uri)
print(f"\nModèle rechargé depuis : {model_uri}")

# Vérification : les prédictions sont identiques
df_verif = model_recharge.transform(df_test.limit(100))
rmse_verif = evaluator_rmse.evaluate(df_verif)
print(f"RMSE sur 100 lignes (vérification) : {rmse_verif:.4f}")


In [ ]:
# Sauvegarde locale du meilleur modèle (format natif Spark)
chemin_model_local = MODELS_DIR / "gbt_best"
cv_model.bestModel.write().overwrite().save(str(chemin_model_local))
print(f"Modèle sauvegardé localement : {chemin_model_local}")

# Pour le recharger plus tard :
# from pyspark.ml import PipelineModel
# model_recharge = PipelineModel.load(str(chemin_model_local))
